# 01 — Data Cleaning

**Input:** `data/raw/sfu_ml.db` 
**Output:** `data/processed/sfu_clean.db` — table: `offerings`

### SFU Section Code Structure

SFU section codes follow the pattern: **[letter][digits]** e.g. `D100`, `E200`, `C100`

The leading letter is the delivery mode:

| Code | Meaning | Primary? |
|------|---------|----------|
| D | Day | ✅ |
| E | Evening | ✅ |
| C | Distance Education | ✅ |
| F | French | ✅ |
| J | SFU NOW | ✅ |
| V | Video conference | ✅ |
| O | Online | ✅ |

Non-primary sections use **multi-letter prefixes**: `LAB`, `TUT`, `STD`, `OPL`, `RQL`, `STL`, `SEM`, `PRA`, `WKS`, `FLD`, `OLC`, `INS`

**Filter rule:** keep rows where `section_code` matches `^[A-Z]\d` — single letter followed by a digit.

### Cleaning Steps
| Step | Filter | Reason |
|------|--------|---------|
| 1 | `section_code` matches `^[A-Z]\d` | Remove labs, tutorials, seminars, practicum etc. |
| 2 | `ml_course_id IS NOT NULL` | Remove sections with no course metadata |
| 3 | `capacity > 0` | Remove placeholders and unconfigured sections |
| 4 | `enrolled IS NOT NULL` | Remove sections with no published enrollment |
| 5 | Drop `waitlist` column | Near-zero variance — no signal for modeling |

In [1]:
import sqlite3
import pandas as pd
from pathlib import Path

RAW_DB   = Path('../data/raw/sfu_ml.db')
CLEAN_DB = Path('../data/processed/sfu_clean.db')
CLEAN_DB.parent.mkdir(exist_ok=True)

assert RAW_DB.exists(), f'not found: {RAW_DB}'
print('ready')

ready


---
## 1. Load raw data

In [2]:
conn = sqlite3.connect(RAW_DB)

raw = pd.read_sql("""
    SELECT
        o.offering_id,
        o.ml_course_id,
        o.ml_term_id,
        o.dept_code,
        o.course_number,
        o.section_code,
        o.instructor,
        o.campus,
        o.capacity,
        o.enrolled,
        o.waitlist,
        c.course_level,
        c.degree_level,
        c.title,
        t.year,
        t.term,
        t.term_order,
        t.semester_code,
        t.is_covid_affected
    FROM ml_section_offerings o
    LEFT JOIN ml_courses c ON o.ml_course_id = c.ml_course_id
    LEFT JOIN ml_terms   t ON o.ml_term_id   = t.ml_term_id
""", conn)

conn.close()
print(f'raw: {len(raw):,} rows  x  {raw.shape[1]} cols')

raw: 50,581 rows  x  19 cols


---
## 2. Understand what section codes actually exist

In [3]:
# extract prefix from each section code
# single-letter codes → D, E, C, F, J, V, O
# multi-letter codes  → LAB, TUT, STD, OPL ...
import re

raw['prefix'] = raw['section_code'].str.extract(r'^([A-Za-z]+)', expand=False).str.upper()
raw['prefix'].value_counts().head(30)

prefix
D      25963
G      16530
I       2193
E       1823
A       1227
OL       987
C        631
B        551
LA       261
F        156
J         84
LB        57
LC        33
P         27
BLS       24
L         19
OP         7
GO         2
OPL        2
LAB        2
HUM        1
Z          1
Name: count, dtype: int64

In [4]:
# confirm which are single-letter (primary) vs multi-letter (non-primary)
raw['is_single_letter'] = raw['prefix'].str.len() == 1
raw.groupby('is_single_letter')['prefix'].value_counts().to_frame('count')

count
is_single_letter prefix       
False            OL        987
                 LA        261
                 LB         57
                 LC         33
                 BLS        24
                 OP          7
                 GO          2
                 LAB         2
                 OPL         2
                 HUM         1
True             D       25963
                 G       16530
                 I        2193
                 E        1823
                 A        1227
                 C         631
                 B         551
                 F         156
                 J          84
                 P          27
                 L          19
                 Z           1

---
## 3. Apply cleaning steps

In [5]:
# Step 1 — primary sections only
# single uppercase letter followed immediately by a digit: D100, E200, C100 etc.
step1 = raw[raw['section_code'].str.match(r'^[A-Z]\d', na=False)]
print(f'after step 1 (primary only):         {len(step1):>7,}  removed {len(raw)-len(step1):,}')

after step 1 (primary only):          49,205  removed 1,376


In [6]:
# confirm what delivery modes survived
step1['prefix'].value_counts()

prefix
D    25963
G    16530
I     2193
E     1823
A     1227
C      631
B      551
F      156
J       84
P       27
L       19
Z        1
Name: count, dtype: int64

In [7]:
# Step 2 — matched courses only
step2 = step1[step1['ml_course_id'].notna()]
print(f'after step 2 (matched courses only):  {len(step2):>7,}  removed {len(step1)-len(step2):,}')

after step 2 (matched courses only):   44,633  removed 4,572


In [8]:
# Step 3 — valid capacity
step3 = step2[(step2['capacity'].notna()) & (step2['capacity'] > 0)]
print(f'after step 3 (capacity > 0):          {len(step3):>7,}  removed {len(step2)-len(step3):,}')

after step 3 (capacity > 0):           44,053  removed 580


In [9]:
# Step 4 — published enrollment
step4 = step3[(step3['enrolled'].notna()) & (step3['enrolled'] > 0)]
print(f'after step 4 (enrolled not null and enrolled > 0):     {len(step4):>7,}  removed {len(step3)-len(step4):,}')

after step 4 (enrolled not null and enrolled > 0):      33,168  removed 10,885


In [10]:
# Step 5 — drop waitlist column
# EDA showed near-zero variance across all sections — no predictive signal
step5 = step4.drop(columns=['waitlist', 'prefix', 'is_single_letter'])

# fix dtypes — pandas reads nullable int columns as float when nulls exist
# they're clean now so convert back to int
step5 = step5.copy()
step5['ml_course_id'] = step5['ml_course_id'].astype(int)
step5['course_level'] = step5['course_level'].astype(int)

print(f'after step 5 (waitlist dropped, dtypes fixed):  {len(step5):>7,}  cols: {step5.shape[1]}')
print()
print(step5[['ml_course_id','course_level']].dtypes)

after step 5 (waitlist dropped, dtypes fixed):   33,168  cols: 18

ml_course_id    int64
course_level    int64
dtype: object


In [11]:
# full summary
clean = step5.copy()

print('=== CLEANING SUMMARY ===')
print(f'Raw rows:                   {len(raw):>8,}')
print(f'Removed — non-primary:      {len(raw)-len(step1):>8,}')
print(f'Removed — unmatched:        {len(step1)-len(step2):>8,}')
print(f'Removed — zero capacity:    {len(step2)-len(step3):>8,}')
print(f'Removed — null enrolled:    {len(step3)-len(step4):>8,}')
print(f'─────────────────────────────────────')
print(f'Clean rows:                 {len(clean):>8,}  ({len(clean)/len(raw)*100:.1f}% retained)')
print(f'Columns:                    {clean.shape[1]:>8}')

=== CLEANING SUMMARY ===
Raw rows:                     50,581
Removed — non-primary:         1,376
Removed — unmatched:           4,572
Removed — zero capacity:         580
Removed — null enrolled:      10,885
─────────────────────────────────────
Clean rows:                   33,168  (65.6% retained)
Columns:                          18


---
## 4. Verify

In [12]:
# no nulls in critical columns
critical = ['ml_course_id', 'ml_term_id', 'dept_code', 'course_number',
            'section_code', 'capacity', 'enrolled', 'course_level']
clean[critical].isnull().sum()

ml_course_id     0
ml_term_id       0
dept_code        0
course_number    0
section_code     0
capacity         0
enrolled         0
course_level     0
dtype: int64

In [13]:
# only single-letter section codes remain
clean['section_code'].str[0].value_counts()

section_code
D    18632
G    10552
E     1611
A      764
C      560
B      498
I      325
F      136
J       58
P       18
L       13
Z        1
Name: count, dtype: int64

In [14]:
# rows per term — summer dip pattern should still be present
(
    clean.groupby(['year', 'term', 'term_order'])
    .size()
    .reset_index(name='n')
    .sort_values(['year', 'term_order'])
    .drop(columns='term_order')
)

,year,term,n
1,2020,spring,2009
2,2020,summer,1324
0,2020,fall,1966
4,2021,spring,2070
5,2021,summer,1322
3,2021,fall,2044
7,2022,spring,2069
8,2022,summer,1380
6,2022,fall,2005
10,2023,spring,2102


In [15]:
# spot check: CMPT 225 across all terms
(
    clean[(clean['dept_code'] == 'CMPT') & (clean['course_number'] == '225')]
    [['year', 'term', 'section_code', 'capacity', 'enrolled', 'instructor']]
    .sort_values(['year', 'term'])
)

,year,term,section_code,capacity,enrolled,instructor
6173,2020,fall,D100,200,157,"Mitchell, David"
6174,2020,fall,D200,200,143,"Shermer, Thomas"
868,2020,spring,D100,184,182,"Mitchell, David"
869,2020,spring,D200,144,115,"Edgar, John"
3711,2020,summer,D100,200,191,"Edgar, John"
3712,2020,summer,D200,150,87,"Taheri Gharagozloo, Pooya"
14712,2021,fall,D100,200,116,"Mitchell, David"
14713,2021,fall,D200,200,140,"Imran, Hazra; Shermer, Thomas"
9216,2021,spring,D100,200,156,"Shinkar, Igor"
9217,2021,spring,D200,160,115,"Thomas, Jack"


In [16]:
# check if any dept+term combinations lost ALL sections after cleaning
# these would be departments that only use seminar/practicum as primary delivery
raw_depts_per_term = raw.groupby(['dept_code', 'year', 'term']).size().reset_index(name='raw_n')
clean_depts_per_term = clean.groupby(['dept_code', 'year', 'term']).size().reset_index(name='clean_n')

merged = raw_depts_per_term.merge(clean_depts_per_term, on=['dept_code','year','term'], how='left')
lost = merged[merged['clean_n'].isna()]

print(f'dept+term combinations that lost all sections: {len(lost)}')
if len(lost) > 0:
    print(lost['dept_code'].value_counts().head(20))

dept+term combinations that lost all sections: 31
dept_code
NUSC     6
SPAN     5
GERM     5
EDPR     4
GRK      3
PERS     2
PLAN     2
SD       2
UGRAD    2
Name: count, dtype: int64


In [17]:
# if any departments lost all their sections, check what section codes they used
# run this only if the cell above shows > 0 lost combinations
if len(lost) > 0:
    lost_depts = lost['dept_code'].unique()
    print('Section codes used by departments that lost all sections:')
    print(
        raw[raw['dept_code'].isin(lost_depts)]
        .groupby(['dept_code', 'prefix'])['offering_id']
        .count()
        .sort_values(ascending=False)
        .head(30)
    )

Section codes used by departments that lost all sections:
dept_code  prefix
EDPR       G         348
NUSC       D          72
GERM       D          67
PLAN       D          57
SD         D          40
SPAN       B          31
           D          25
GRK        B          19
           D          14
UGRAD      D          14
SD         E          10
SPAN       L           9
SD         OL          9
PERS       D           6
PLAN       E           6
           B           5
GERM       L           3
PLAN       OL          2
SD         B           2
           C           2
GRK        L           1
Name: offering_id, dtype: int64


---
## 5. Save to processed DB

In [18]:
if CLEAN_DB.exists():
    CLEAN_DB.unlink()

conn = sqlite3.connect(CLEAN_DB)

# create table with explicit schema so SQLite respects types and PK
conn.execute("""
    CREATE TABLE offerings (
        offering_id       INTEGER PRIMARY KEY,
        ml_course_id      INTEGER NOT NULL,
        ml_term_id        INTEGER NOT NULL,
        dept_code         TEXT,
        course_number     TEXT,
        section_code      TEXT,
        instructor        TEXT,
        campus            TEXT,
        capacity          INTEGER,
        enrolled          INTEGER,
        course_level      INTEGER,
        degree_level      TEXT,
        title             TEXT,
        year              INTEGER,
        term              TEXT,
        term_order        INTEGER,
        semester_code     INTEGER,
        is_covid_affected INTEGER
    )
""")

# append into the pre-created table — preserves schema and PK
clean.to_sql('offerings', conn, index=False, if_exists='append')
conn.commit()
conn.close()

print(f'saved → {CLEAN_DB}')
print(f'rows:   {len(clean):,}')
print(f'cols:   {clean.shape[1]}')

saved → ..\data\processed\sfu_clean.db
rows:   33,168
cols:   18


In [19]:
# verify the saved DB reads back correctly
conn  = sqlite3.connect(CLEAN_DB)
check = pd.read_sql('SELECT COUNT(*) as n FROM offerings', conn)
cols  = pd.read_sql('PRAGMA table_info(offerings)', conn)
conn.close()

print(f'rows in sfu_clean.db: {check["n"].iloc[0]:,}')
print()
print('columns saved:')
print(cols[['name', 'type']].to_string(index=False))

rows in sfu_clean.db: 33,168

columns saved:
             name    type
      offering_id INTEGER
     ml_course_id INTEGER
       ml_term_id INTEGER
        dept_code    TEXT
    course_number    TEXT
     section_code    TEXT
       instructor    TEXT
           campus    TEXT
         capacity INTEGER
         enrolled INTEGER
     course_level INTEGER
     degree_level    TEXT
            title    TEXT
             year INTEGER
             term    TEXT
       term_order INTEGER
    semester_code INTEGER
is_covid_affected INTEGER
